# Challenge: Pareto Optimization of SEI Additives with ALCHEMI

This challenge extends the Part 1 batched adsorption tutorial to a simplified battery-interface problem. You will run a compact ALCHEMI Toolkit relaxation workflow, compute binding energies, convert those energies into two challenge metrics, and select the additive that gives the largest Pareto hypervolume improvement over baseline electrolyte solvents.

The chemistry here is intentionally simplified. Li metal is a reactive anode proxy. Each molecule class maps to one passivating SEI-product proxy surface using the lookup table in `data/class_surface_lookup.csv`. The bundled structures are teaching inputs for a workflow exercise, not production battery-interface reference models.


## What You Need To Produce

Write `outputs/challenge_submission.csv` with one row per molecule and these columns:

`candidate_id`, `role`, `molecule_class`, `passivating_surface_id`, `E_bind_Li_eV`, `E_bind_passivating_eV`, `seeding_score`, `passivation_score`, `is_pareto`, `hypervolume_improvement`, `selected`.

Optional but recommended: also write `outputs/raw_component_energies.csv` so the grader can check your binding-energy arithmetic without running any model calls.


## Control Panel

These defaults mirror the Part 1 tutorial but keep the challenge small. You may reduce `TOOLKIT_N_STEPS` while debugging, then rerun with the default before submitting.


In [ ]:
from pathlib import Path

TOOLKIT_CHECKPOINT = "medium-mpa-0"
TOOLKIT_HEAD = None
TOOLKIT_DEVICE = "auto"
TOOLKIT_DTYPE = "float32"
TOOLKIT_COMPILE_MODEL = False
TOOLKIT_ENABLE_CUEQ = True
TOOLKIT_DT = 0.01
TOOLKIT_N_STEPS = 300
TOOLKIT_FMAX = 0.10
TOOLKIT_D3BJ = None
BATCH_SIZE = 4

ADSORPTION_HEIGHT_A = 2.6
FROZEN_SURFACE_FRACTION = 0.5

OUTPUT_DIR = Path("outputs")
SUBMISSION_PATH = OUTPUT_DIR / "challenge_submission.csv"
RAW_COMPONENT_ENERGIES_PATH = OUTPUT_DIR / "raw_component_energies.csv"


## Setup

The [ALCHEMI Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/) describes the same core workflow used in Part 1: structures are represented as `AtomicData`, packed into `Batch` objects, evaluated by model wrappers such as MACE, and relaxed with FIRE2. This notebook reuses the Part 1 helper backend so your challenge code focuses on the scientific workflow and bookkeeping.


In [ ]:
import os
import sys
from importlib.metadata import version, PackageNotFoundError

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "data" / "molecule_manifest.csv").exists():
    candidate = NOTEBOOK_DIR / "challenge-sei"
    if (candidate / "data" / "molecule_manifest.csv").exists():
        NOTEBOOK_DIR = candidate.resolve()
    else:
        raise RuntimeError("Start Jupyter from challenge-sei or from the repository root.")
os.chdir(NOTEBOOK_DIR)

REPO_ROOT = NOTEBOOK_DIR.parent
PART1_ROOT = REPO_ROOT / "part-1-batched-adsorption"
if not (PART1_ROOT / "helpers" / "__init__.py").exists():
    raise RuntimeError("Cannot find Part 1 helpers. Keep challenge-sei beside part-1-batched-adsorption.")
sys.path.insert(0, str(PART1_ROOT))

import numpy as np
import pandas as pd
from ase.io import read as ase_read
from ase.io import write as ase_write

from helpers import (
    ToolkitRelaxationConfig,
    ToolkitD3BJConfig,
    check_toolkit_native_api,
    get_toolkit_relaxation_engine,
    ase_to_atomic_data,
    atomic_data_to_ase,
    make_active_mask,
)

print(f"Challenge folder : {NOTEBOOK_DIR.name}")
print(f"Part 1 helpers   : {PART1_ROOT.relative_to(REPO_ROOT)}")
for pkg in ("ase", "numpy", "pandas", "torch", "nvalchemi-toolkit"):
    try:
        print(f"{pkg:<18}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<18}: not installed")


## 1. Load The Challenge Manifests

Fill in this cell so every molecule has its class-specific passivating surface. Keep the baseline rows (`EC`, `EMC`) because they define the starting Pareto front.


In [ ]:
# TODO: Load data/molecule_manifest.csv, data/surface_manifest.csv, and
# data/class_surface_lookup.csv with pandas. Merge the molecule table with the
# lookup table on molecule_class so each molecule has passivating_surface_id.
# Store the merged table in challenge_df.

# molecules_df = ...
# surfaces_df = ...
# lookup_df = ...
# challenge_df = ...

raise NotImplementedError("Load and merge the challenge manifests.")

# Suggested checks after you implement the merge:
# assert challenge_df["passivating_surface_id"].notna().all()
# assert set(challenge_df["role"]) == {"baseline", "additive"}
# display(challenge_df[["candidate_id", "role", "molecule_class", "passivating_surface_id"]])


## 2. Build The Toolkit Relaxation Engine

This is the same native Toolkit path used in Part 1. D3 is disabled by default for a compact challenge run; if you enable it for an instructor rerun, record the parameters in your notes.


In [ ]:
status = check_toolkit_native_api()
print(status["message"])
if not status["available"]:
    raise RuntimeError("ALCHEMI Toolkit native API is not available in this kernel.")

if isinstance(TOOLKIT_D3BJ, dict):
    TOOLKIT_D3BJ = ToolkitD3BJConfig(**TOOLKIT_D3BJ)

relaxation_config = ToolkitRelaxationConfig(
    name="toolkit",
    cache_dir=(OUTPUT_DIR / "cache_json").as_posix(),
    use_cached_responses=False,
    toolkit_checkpoint=TOOLKIT_CHECKPOINT,
    toolkit_head=TOOLKIT_HEAD,
    toolkit_device=TOOLKIT_DEVICE,
    toolkit_dtype=TOOLKIT_DTYPE,
    toolkit_compile_model=TOOLKIT_COMPILE_MODEL,
    toolkit_enable_cueq=TOOLKIT_ENABLE_CUEQ,
    toolkit_dt=TOOLKIT_DT,
    toolkit_n_steps=TOOLKIT_N_STEPS,
    toolkit_fmax=TOOLKIT_FMAX,
    toolkit_d3bj=TOOLKIT_D3BJ,
    toolkit_require_d3bj=TOOLKIT_D3BJ is not None,
)
RELAXATION_ENGINE = get_toolkit_relaxation_engine(relaxation_config)
print(f"Toolkit relaxation engine ready: {RELAXATION_ENGINE.name}")


## 3. Structure Helpers

These helpers load the bundled structures and place each molecule above the center of a teaching slab. The placement is intentionally minimal: the challenge is about reproducing the Part 1 workflow logic, not building a production adsorption-site search.


In [ ]:
def load_atoms(relative_path):
    atoms = ase_read(Path(relative_path))
    return atoms


def gas_box(atoms, *, box_A=15.0):
    gas = atoms.copy()
    gas.set_cell([box_A, box_A, box_A])
    gas.set_pbc([True, True, True])
    gas.center()
    return gas


def surface_center_xy(surface):
    cell = np.asarray(surface.cell.array, dtype=float)
    center = 0.5 * (cell[0] + cell[1])
    return center[:2]


def place_molecule_on_surface(surface, molecule, *, height_A=ADSORPTION_HEIGHT_A):
    slab = surface.copy()
    mol = molecule.copy()
    mol.translate(-mol.get_center_of_mass())
    top_z = float(np.max(slab.positions[:, 2]))
    bottom_z = float(np.min(mol.positions[:, 2]))
    xy = surface_center_xy(slab)
    mol.translate([xy[0], xy[1], top_z + height_A - bottom_z])
    combined = slab + mol
    combined.set_cell(slab.cell)
    combined.set_pbc(slab.pbc)
    return combined


def combined_active_mask(surface, combined):
    surface_mask = make_active_mask(surface, bottom_fraction=FROZEN_SURFACE_FRACTION)
    return surface_mask + [True] * (len(combined) - len(surface))


def relax_structures(jobs, *, batch_size=BATCH_SIZE, label_prefix="sei_challenge"):
    rows = []
    for start in range(0, len(jobs), batch_size):
        chunk = jobs[start:start + batch_size]
        payloads = [
            ase_to_atomic_data(job["atoms"], structure_id=job["job_id"], active_mask=job.get("active_mask"))
            for job in chunk
        ]
        reply = RELAXATION_ENGINE.relax(payloads, label=f"{label_prefix}_{start // batch_size + 1:03d}")
        for job, result in zip(chunk, reply.atoms):
            rows.append({
                **{key: value for key, value in job.items() if key not in {"atoms", "active_mask"}},
                "energy_eV": float(result.energy),
                "converged": bool(result.converged),
                "optimizer_nsteps": int(result.num_optimization_steps),
                "relaxed_atoms": atomic_data_to_ase(result),
            })
    return rows


## 4. Build And Relax The Jobs

You need three groups of energies:

1. Gas molecules: `E_species`.
2. Clean surfaces: `E_surface` for `Li_metal` and every passivating surface used by the candidates.
3. Combined systems: `E_surface+species` for molecule on Li metal and molecule on its class-specific passivating surface.


In [ ]:
# TODO: Build gas_jobs, clean_surface_jobs, and combined_jobs, then relax them
# with relax_structures(...). Keep enough metadata in each job to recover the
# candidate_id, interaction, and surface_id after relaxation.
#
# Hints:
# - load molecules from challenge_df["structure_path"].
# - load surfaces from surfaces_df["structure_path"].
# - use gas_box(...) for isolated molecule references.
# - use place_molecule_on_surface(...) for combined systems.
# - use combined_active_mask(surface, combined) for surface+molecule jobs.
# - clean surfaces should use make_active_mask(surface, bottom_fraction=FROZEN_SURFACE_FRACTION).

# gas_results = relax_structures(gas_jobs, label_prefix="sei_gas")
# clean_surface_results = relax_structures(clean_surface_jobs, label_prefix="sei_clean_surface")
# combined_results = relax_structures(combined_jobs, label_prefix="sei_combined")

raise NotImplementedError("Build and relax the gas, clean-surface, and combined jobs.")


## 5. Compute Binding Energies

Use the same convention as Part 1:

`E_bind = E_surface+species - E_surface - E_species`

Negative values mean exothermic binding in this challenge convention.


In [ ]:
# TODO: Convert gas_results, clean_surface_results, and combined_results into
# lookup dictionaries, then compute one raw-energy row for each candidate on
# Li_metal and one raw-energy row for the candidate's passivating surface.
# Store the result in raw_component_energies_df with columns:
# candidate_id, interaction, surface_id, E_surface_species_eV, E_surface_eV, E_species_eV

# raw_component_energies_df = ...

raise NotImplementedError("Compute the raw component energy table.")

# OUTPUT_DIR.mkdir(exist_ok=True)
# raw_component_energies_df.to_csv(RAW_COMPONENT_ENERGIES_PATH, index=False)
# display(raw_component_energies_df.head())


In [ ]:
# TODO: Use raw_component_energies_df to compute E_bind_Li_eV and
# E_bind_passivating_eV for every row in challenge_df. Store the table in
# binding_df and keep the required metadata columns.

# binding_df = ...

raise NotImplementedError("Compute binding energies from component energies.")

# display(binding_df[["candidate_id", "role", "E_bind_Li_eV", "E_bind_passivating_eV"]])


## 6. Compute Scores

The seeding score rewards moderate interaction with reactive Li metal. The passivation score rewards weak interaction with the passivating SEI proxy.


In [ ]:
def seeding_score(e_bind_li_eV):
    # TODO: implement max(0, 1 - abs(E_bind_Li - (-1.0)) / 1.0)
    raise NotImplementedError


def passivation_score(e_bind_passivating_eV):
    # TODO: implement clip((E_bind_passivating + 1.0) / 1.0, 0, 1)
    raise NotImplementedError


# TODO: Add seeding_score and passivation_score columns to binding_df.
# scored_df = ...

raise NotImplementedError("Compute challenge scores.")


## 7. Pareto Front And Hypervolume Improvement

Treat both scores as objectives to maximize. The baseline front is built from `EC` and `EMC`. For each additive, compute how much the 2D dominated hypervolume increases when that additive is added to the baseline front. Use reference point `(0, 0)`.


In [ ]:
def dominates(a, b):
    # TODO: return True if point a dominates point b for 2D maximization.
    raise NotImplementedError


def pareto_flags(points):
    # TODO: return a list of booleans marking nondominated points.
    raise NotImplementedError


def hypervolume_2d(points):
    # TODO: compute 2D dominated area against reference point (0, 0).
    # Hint: filter to the Pareto front, sort by seeding score, then sum
    # rectangular slices.
    raise NotImplementedError


# TODO: Add is_pareto and hypervolume_improvement columns to scored_df.
# final_df = ...

raise NotImplementedError("Compute Pareto flags and hypervolume improvements.")


## 8. Select Your Additive And Submit

Mark exactly one additive as `selected=True`: the additive with the maximum hypervolume improvement. Baseline rows should not be selected.


In [ ]:
# TODO: Create a boolean selected column in final_df. Exactly one additive should
# be True, and it should have the maximum hypervolume_improvement.
# submission = final_df[[
#     "candidate_id", "role", "molecule_class", "passivating_surface_id",
#     "E_bind_Li_eV", "E_bind_passivating_eV", "seeding_score",
#     "passivation_score", "is_pareto", "hypervolume_improvement", "selected",
# ]].copy()

raise NotImplementedError("Select the final additive and build the submission table.")


In [ ]:
required_columns = [
    "candidate_id", "role", "molecule_class", "passivating_surface_id",
    "E_bind_Li_eV", "E_bind_passivating_eV", "seeding_score",
    "passivation_score", "is_pareto", "hypervolume_improvement", "selected",
]
missing = [column for column in required_columns if column not in submission.columns]
if missing:
    raise RuntimeError(f"Submission is missing required columns: {missing}")
if int(submission["selected"].sum()) != 1:
    raise RuntimeError("Exactly one row must be selected.")

OUTPUT_DIR.mkdir(exist_ok=True)
submission[required_columns].to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}")
display(submission[required_columns])
